# YouTube Comments Scraper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mamoo85/open-data-actor-notebooks/blob/main/notebooks/youtube-comments-scraper.ipynb)

**What:** Scrape public YouTube comments and replies from videos, Shorts, channels, playlists or searches: text, author, likes, reply count, creator heart, pinned, date. Top or newest, keyword/likes/date filters, monitor mode. No login.

**Why:** run the [`youtube-comments-scraper`](https://apify.com/transparent_meteorite/youtube-comments-scraper) Apify actor from Python and analyse the result as a pandas DataFrame, with no scraping code to maintain.

> **Disclosure:** I am the author of this actor. It is pay-per-result; the free plan credit on a new Apify account covers a trial run like this one (this notebook uses a very small sample input).

**Privacy:** Commenter names, handles and channel links are dropped before display; comment text and engagement counts are shown.

## 1. Setup

In [1]:
!pip install -q apify-client pandas

import os, getpass
try:
    from google.colab import userdata
    APIFY_TOKEN = userdata.get('APIFY_TOKEN')          # Colab: add it under the key icon (Secrets)
except Exception:
    APIFY_TOKEN = os.environ.get('APIFY_TOKEN') or getpass.getpass('Apify API token: ')
assert APIFY_TOKEN, 'An Apify API token is required (a free account is enough).'
print('Token loaded.')

Token loaded.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Input
A deliberately small sample so the run is cheap and quick. See the [actor page](https://apify.com/transparent_meteorite/youtube-comments-scraper) for every option.

In [2]:
run_input = {   'startUrls': [{'url': 'https://www.youtube.com/watch?v=dQw4w9WgXcQ'}],
    'maxComments': 5,
    'sortBy': 'TOP',
    'includeReplies': False}

## 3. Run the actor and read the dataset

In [3]:
from apify_client import ApifyClient

client = ApifyClient(APIFY_TOKEN)
run = client.actor('transparent_meteorite/youtube-comments-scraper').call(run_input=run_input)
# apify-client v2 returns a dict, v3 a Run object; handle both
get = (lambda k, a: run[k]) if isinstance(run, dict) else (lambda k, a: getattr(run, a))
items = list(client.dataset(get('defaultDatasetId', 'default_dataset_id')).iterate_items())
print('Run status:', get('status', 'status'), '| items:', len(items))

[apify.youtube-comments-scraper runId:CNVgfVcZRtFIt66me] -> Status: RUNNING, Message: 


[apify.youtube-comments-scraper runId:CNVgfVcZRtFIt66me] -> 2026-10-14T07:15:45.661Z ACTOR: Pulling container image of build jaXytFetCamRu8D6w from registry.


[apify.youtube-comments-scraper runId:CNVgfVcZRtFIt66me] -> 2026-10-14T07:15:45.693Z ACTOR: Creating container.


[apify.youtube-comments-scraper runId:CNVgfVcZRtFIt66me] -> 2026-10-14T07:15:45.759Z ACTOR: Starting container.


[apify.youtube-comments-scraper runId:CNVgfVcZRtFIt66me] -> 2026-10-14T07:15:45.760Z ACTOR: Running under "LIMITED_PERMISSIONS".


...

[apify.youtube-comments-scraper runId:CNVgfVcZRtFIt66me] -> 2026-10-14T07:15:48.269Z Done. 5 comment rows (0 replies, charged 5) from 1 video(s); comments off 0, unavailable 0, failed 0, filtered 0; requests 2, blocked 0.


[apify.youtube-comments-scraper runId:CNVgfVcZRtFIt66me] -> 2026-10-14T07:15:48.344Z INFO  [Status message]: Done: 5 items


[apify.youtube-comments-scraper runId:CNVgfVcZRtFIt66me] -> 2026-10-14T07:15:48.382Z Thanks for using YouTube Comments Scraper. Found a bug or need a field? Open an issue on the Issues tab (https://console.apify.com/actors/u5lldCrepOymI4dy7/issues) with the run URL and we will reply there.


[apify.youtube-comments-scraper runId:CNVgfVcZRtFIt66me] -> Status: SUCCEEDED, Message: Done: 5 items


Run status: SUCCEEDED | items: 5


## 4. Tabulate

Commenter names, handles and channel links are dropped before display; comment text and engagement counts are shown.

In [4]:
import pandas as pd
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 200)

df = pd.DataFrame(items)

# Privacy: remove contact / personal-data columns before anything is displayed
DROP = ["email", "phone", "author", "registeredagent", "contractor", "officer", "linkedin", "facebook", "instagram", "youtube", "tiktok", "socials", "twitter"]
KEEP = ["emailCount", "phoneCount", "emailProvider"]
df = df[[c for c in df.columns if c in KEEP or not any(p in c.lower() for p in DROP)]]
SHOW = [c for c in ["videoTitle", "text", "likeCount", "replyCount", "isReply", "hasCreatorHeart", "isPinned", "publishedAtApprox"] if c in df.columns]
df[SHOW]

,videoTitle,text,likeCount,replyCount,isReply,hasCreatorHeart,isPinned,publishedAtApprox
0,Rick Astley - Never Gonna Give You Up (Official Video) (4K Remaster),"Let’s put the meme aside, this is actually a banging song.",219000,590,False,False,False,2020-10-13T19:15:47.932Z
1,Rick Astley - Never Gonna Give You Up (Official Video) (4K Remaster),1987 : normal song \r\n2026 : national anthem of the universe,138000,933,False,False,False,2020-10-13T19:15:47.932Z
2,Rick Astley - Never Gonna Give You Up (Official Video) (4K Remaster),Gonna flag this for nudity so I can rick roll the YouTube staff,568000,714,False,False,False,2020-10-13T19:15:47.932Z
3,Rick Astley - Never Gonna Give You Up (Official Video) (4K Remaster),I scaned a QR code,259,15,False,False,False,2026-10-12T07:15:47.932Z
4,Rick Astley - Never Gonna Give You Up (Official Video) (4K Remaster),Ok let’s be honest here\nRick rolling is the one meme that will ne...,427000,730,False,False,False,2020-10-13T19:15:47.932Z


## 5. A quick analysis

In [5]:
for c in ['likeCount','replyCount']:
    df[c]=pd.to_numeric(df[c],errors='coerce')
print('Comments:',len(df),'| mean likes:',round(df['likeCount'].mean(),1))
df.sort_values('likeCount',ascending=False)[['text','likeCount','replyCount']].head(5)

Comments: 5 | mean likes: 270451.8


,text,likeCount,replyCount
2,Gonna flag this for nudity so I can rick roll the YouTube staff,568000,714
4,Ok let’s be honest here\nRick rolling is the one meme that will ne...,427000,730
0,"Let’s put the meme aside, this is actually a banging song.",219000,590
1,1987 : normal song \r\n2026 : national anthem of the universe,138000,933
3,I scaned a QR code,259,15


## Next steps

- Raise the limits in the input cell to run a larger sample.
- Schedule the actor or call it from your own pipeline: [youtube-comments-scraper on the Apify Store](https://apify.com/transparent_meteorite/youtube-comments-scraper).
- Check the Store page for the current pay-per-result price before scaling up.

Generated by `tools/generate.py`.